# Introduction to Singular Value Decomposition.

SVD stands for Singular Value Decompostion and is very vastly used in the field of computation engineering and machine learning for feature extraction, linear regression problems with least squares, dimension reduction, etc.

According to wikipedia "*In linear algebra, the singular value decomposition (SVD) is a factorization of a real or complex matrix. It generalizes the eigendecomposition of a square normal matrix with an orthonormal eigenbasis to any {\displaystyle m\times n}m\times n matrix. It is related to the polar decomposition*". It insinuates that a given matrix M can be decomposed into three different matrices U, Σ and V\* such that  ![](https://wikimedia.org/api/rest_v1/media/math/render/svg/cb961f1aabc39aba6b7c00c8db240ac74ff2d7a2). Where V\* is the transpose of matrix V, V and U are orthognal matrices of a matrice of numbers M and Σ is a diagonal matrix.

For a given matrix M of size m x n, the corresponding U is a matrix of size m x m, Σ is a matrix of size m x n and V is a matrix of size n x n.

To compute U, Σ and V, we need to compute the covariance / correlation matrices of matrix M row wise and column wise. the resultant matrices of these respective operations will yield square matrices. Eg: if matrix M is of size 300 x 100, then to compute a row wise correlation, we determine M* (M transpose), where  M* will be of size 100 x 300 and hence (M x M*) will be of size 300 x 300 and this resultant matrix is the **row wise correlation matrix of M**. Similarly, (M* x M) will be of size 100 x 100 and this corresponds to the **column wise correlation matrix of M**. The Eigenvectors of (M x M*) and (M* x M) respectively are U and V in the singular value decomposition, and Σ is a diagonal martix with each value in the diagonal corresponding to the Eigenvalues.

For more information, visit <a href='https://en.wikipedia.org/wiki/Singular_value_decomposition'>here.</a>

# Testing the SVD algorithm

In this implementation SVD is used on a high resolution image for dimension reduction. The image has 3 color channels RGB and the following steps were carried out:

1. The color channels are seperated into 3 different matrices.

2. Their respective SVD matrices are computed.

3. A rank value r is selected such that matrices U', Σ' and V\*' are of sizes m x r, r x r and r x n respectively. The matrix product of these matrices are the rank r approximation of the image of orignal decomposed matrices U, Σ and V\*, where "r" simply indicates the index of singular values and singular vectors.

4. The matrices are multiplied to give an approximate version of the original RGB channel matrices which are then stacked to give back an approximate version of the original image.

Below we define two functions *stack()* and *SVD()* for performing the following steps above.

The SVD function returns the approximated version of the original image, the Eigenvalues Σ of all the RGB channel matrices, and the memory consumption of matrices U', Σ' and V\*'

In [1]:
from matplotlib import pyplot as plt
from jax import numpy as jnp
import tqdm

def stack(matrices):
    return jnp.dstack([i for i in matrices])

def SVD(matrices, rank):
    approx_matrices = []
    sigmas = []
    memory_sizes = []

    for idx, matrix in enumerate(matrices):
        U, S, Vt = jnp.linalg.svd(matrix, full_matrices=False)
        sigmas.append(S)
        S = jnp.diag(S)
        truncated_U,  truncated_S, truncated_V = U[:, :rank],  S[:rank, :rank],  Vt[:rank, :]
        memory_sizes.append(truncated_U.nbytes + truncated_S.nbytes + truncated_V.nbytes)
        approx_matrices.append(truncated_U @ truncated_S @ truncated_V)

    total_size = sum(memory_sizes)
    return stack(approx_matrices), sigmas, total_size

Below, we visualise the original image, determine it's memory, compute and store the Frobenius norm of the image matrices.

The Frobenius norm of a matrix is simply the two norm of that matrix stretched into a single vector. According to Ekard Young theorem, the best approximation of a given matrix M is simply the first r truncted Eigenvalues and Eigenvectors of the singular value decomposition. What this says in essence is that the best value that minimizes the difference between Frobenius norm of the original and approximated matrix is given by the matrix product of U', Σ' and V\*' from the above steps.

In [2]:
try:
  img = plt.imread('/content/Logo-unah.jpg')
  original_img_size = img.nbytes
  img_frob_norm = jnp.linalg.norm(img)

  plt.figure(figsize = (10, 5))
  plt.imshow(img)
  plt.title('Original Image')
except FileExistsError:
  print("la imagen no existe en la direccion dada")


FileNotFoundError: [Errno 2] No such file or directory: '/content/Logo-unah.jpg'

In [ ]:
img_red = img[:, :, 0]
img_green = img[:, :, 1]
img_blue = img[:, :, 2]

Below we define a list of rank r values (10, 20, 30, ...100), iterate through the list using each r value for a rank r approximation of the image, compute the memory compression rate of the image, compute the difference between Frobenius norms of the original and approximated image and plot the image for a given r values for visual represention.

In [ ]:
ranks = range(10, 110, 10)

fig, axs = plt.subplots(5, 2, figsize=(20,20))

for idx, r in tqdm.tqdm(enumerate(ranks)):
    approx_img, sigmas, memory_size = SVD([img_red, img_green, img_blue], rank=r)

    approx_img = jnp.array(approx_img, dtype=jnp.uint8)
    norm_diff = img_frob_norm - jnp.linalg.norm(approx_img)
    compression_percentage = ((original_img_size - memory_size) / original_img_size) * 100

    x = idx-5 if (idx > 4) else idx
    y = 0 if (idx < 5) else 1
    axs[x, y].imshow(approx_img)
    axs[x, y].set_title(f'\n\n\nRank = {r} with compression rate of {round(compression_percentage, 3)}% and norm difference {round(norm_diff, 3)}')

In [ ]:
import ipywidgets as widgets
from IPython.display import display

def update_svd(rank):
    # Reconstruct the image with the specified rank
    approx_img, _, memory_size = SVD([img_red, img_green, img_blue], rank=rank)
    approx_img = jnp.array(approx_img, dtype=jnp.uint8)

    # Calculate metrics
    norm_diff = img_frob_norm - jnp.linalg.norm(approx_img)
    compression_percentage = ((original_img_size - memory_size) / original_img_size) * 100

    # Plot
    plt.figure(figsize=(10, 6))
    plt.imshow(approx_img)
    plt.title(f'Rank = {rank}\nCompression Rate = {compression_percentage:.3f}%\nNorm Difference = {norm_diff:.3f}')
    plt.axis('off')
    plt.show()

# The maximum rank is constrained by the minimum dimension of the image channels
max_rank = min(img_red.shape)

# Create interactive slider
widgets.interact(
    update_svd,
    rank=widgets.IntSlider(min=1, max=max_rank, step=1, value=50, description='Rank r:')
)

From the above plot, we can see that at r = 10, the approximated image is blurred and the memory usage of matrices U', Σ' and V\*' for all channels combined is 94% lesser than the orignal. Similarly, at r = 20, the approximated image is slightly better than the previous one, with 88% less memory than the original image. At rank=100, we got an almost perfect approximation of the image, with 40% less memory than the original image.

Below, we plot the Eigenvalues Σ from the decomposition of all the different channel matrices on a log plot. This plot aids us to indicates what rank or set of singluar / eigenvalues hold most information and what rank or set of singluar / eigenvalues hold least information pertaining to the channel matrices.

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(20, 5))
channels = ['red', 'green', 'blue']
for idx, S in enumerate(sigmas):
    axs[idx].semilogy(S, c=channels[idx])
    axs[idx].set_ylabel('information energy')
    axs[idx].set_xlabel('rank r')
    axs[idx].set_title(f'Eigenvalues log plot for {channels[idx]} channel')


From the three different plots, we can inspect that the rank of the singular values that hold most information pertaining the matrices seem to be even less than 100. This plot is very useful for selecting the appropriate r value for truncating the decomposition matrices for dimension reduction.

Similar to the plot above, the plot below also tells us what rank value or set of singular values hold most or least of the information pertaining to the image. The plot tells us the percentage of information (on the y-axis) held by a given rank value (on the x-axis).

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(20, 5))
channels = ['red', 'green', 'blue']
for idx, S in enumerate(sigmas):
    sigma_cumsum = jnp.cumsum(S)
    sigma_sum = jnp.sum(S)
    axs[idx].semilogy(sigma_cumsum/sigma_sum, c=channels[idx])
    axs[idx].set_title(f'Eigenvalues log plot for {channels[idx]} channel')

NB: *SVDs can be used to accertain the principle components of turbulent flow that happens to have very high degree of flow freedom, hence reducing the overall dimension of the flow data for modeling.*